In [ ]:
# z zalozeniem t-studenta
import hashlib
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from arch.bootstrap import StationaryBootstrap
from scipy.stats import norm
from scipy.stats import t as t_dist
from sklearn.linear_model import QuantileRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Parametry globalne
# ---------------------------------------------------------------------------
alphas   = [0.01, 0.025, 0.05]   # poziomy VaR: regulacyjny 1% + 2,5%/5% dla mocy testow
W        = 750                   # okno ~3 lata handlowe (kompromis stabilnosc/responsywnosc)
B        = 5000                  # replikacje bootstrap: blad Monte Carlo mediany ~1/sqrt(B) ~1,4%
N        = 100_000               # scenariusze MC: ~1000 obs. ponizej kwantyla 1% -> pomijalna wariancja
scale    = 100                   # przeskalowanie stop dla uwarunkowania optymalizacji GARCH
BLOCK_LEN = 20   # sr. dlugosc bloku bootstrapu stacjonarnego ~ 1 miesiac handlowy (klastrowanie zmiennosci)
tickers  = ["AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS",
            "GS", "HD", "HON", "IBM", "JNJ", "JPM", "KO", "MCD", "MMM", "MRK",
            "MSFT", "NKE", "NVDA", "PG", "SHW", "TRV", "UNH", "V", "VZ", "WMT"]
N_JOBS   = -1   # joblib: -1 = wszystkie rdzenie (zrownoleglenie po tickerach)

# --- Tryb szybkiego podgladu (QUICK): mala probka do walidacji logiki. ---
# Ustaw QUICK = False dla PELNEGO przebiegu (30 tickerow, W=750, B=5000, 2010-2026).
QUICK = False
if QUICK:
    tickers = ["AAPL", "MSFT", "JPM"]
    W = 250
    B = 200
    N = 5000
    DATE_START, DATE_END = "2015-01-01", "2019-01-01"
else:
    DATE_START, DATE_END = "2010-01-01", "2026-01-01"

lookback     = 20
VAL_FRAC     = 0.25
N_TRAIN_MIN  = 15
N_VAL_MIN    = 5
grid_alpha_qr = [0.001, 0.01, 0.1, 1.0]

# ---------------------------------------------------------------------------
# STALE df dla t-Studenta (metody z grubymi ogonami: MC, GARCH, ARMA-GARCH)
# ---------------------------------------------------------------------------
# Uzasadnienie wyboru df=5:
# - empiryczne stopy zwrotu akcji maja ogony GRUBSZE niz rozklad normalny
#   (leptokurtoza) - dobrze udokumentowany fakt stylizowany rynkow finansowych,
#   podkreslany m.in. w pracach K. Piontka nt. kwantylowych modeli ryzyka.
# - df=5 daje kurtoze = 3 + 6/(df-4) = 9, tj. grube ogony w gornej czesci
#   zakresu obserwowanego empirycznie dla akcji (kurtoza ~4-8).
# - df < 3 -> nieskonczona wariancja (nierealistyczne); df > 2 gwarantuje
#   skonczona wariancje; df > 10 -> rozklad zbliza sie do normalnego.
# - wartosc STALA (a nie estymowana z okna) zapewnia stabilnosc i
#   porownywalnosc prognoz miedzy oknami i miedzy 30 spolkami DJIA.
# UWAGA: metoda VC celowo uzywa rozkladu NORMALNEGo (klasyczny delta-normal)
# jako "naiwnego" benchmarku - kontrast z metodami t(5) pokazuje koszt
# ignorowania grubych ogonow.
#
# Kluczowy szczegół: t(df) ma wariancję = df/(df-2), a nie 1.
# Oznacza to, że jeśli chcemy zachować tę samą wariancję co dane (σ²),
# musimy przeskalować rozkład: scale_t = σ / sqrt(df/(df-2))
# Wtedy: Var(scale_t * T) = scale_t² * df/(df-2) = σ²  ✓
#
# W kodzie stosujemy stały współczynnik korekcji:
T_DF = 5
T_VAR_CORRECTION = np.sqrt(T_DF / (T_DF - 2))   # = sqrt(5/3) ≈ 1.291
# Przeskalowanie sigma: scale_t = sigma / T_VAR_CORRECTION
# Kwantyl VaR: z_t = t_dist.ppf(alpha, df=T_DF) / T_VAR_CORRECTION
# (równoważnie: -(mu + sigma * z_t_standaryzowany))

# Prekalkulowany standaryzowany kwantyl dla każdego alpha
# (stały przez całe okno kroczące — nie przeliczamy w pętli)
Z_T5 = {a: t_dist.ppf(a, df=T_DF) / T_VAR_CORRECTION for a in alphas}
# Kwantyl rozkladu NORMALNEGO dla metody VC (klasyczny delta-normal / RiskMetrics)
Z_NORM = {a: norm.ppf(a) for a in alphas}
# Przykładowe wartości:
# alpha=0.01:  z ≈ -2.906 (vs norm.ppf(0.01) ≈ -2.326) → bardziej konserwatywny VaR
# alpha=0.025: z ≈ -2.302 (vs norm.ppf(0.025) ≈ -1.960)
# alpha=0.05:  z ≈ -1.857 (vs norm.ppf(0.05)  ≈ -1.645)


def make_seed(base: int, t: int) -> int:
    h = hashlib.md5(f"{base}_{t}".encode()).hexdigest()
    return int(h, 16) % (2 ** 31)


def pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, alpha: float) -> float:
    e = y_true - y_pred
    return float(np.mean(np.maximum(alpha * e, (alpha - 1.0) * e)))


def build_supervised(window_returns: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(lookback, len(window_returns)):
        X.append(window_returns[i - lookback:i])
        y.append(window_returns[i])
    return np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.float64)


def split_sizes(n_total: int):
    n_val   = max(N_VAL_MIN, int(VAL_FRAC * n_total))
    n_train = n_total - n_val
    if n_train < N_TRAIN_MIN:
        n_train = N_TRAIN_MIN
        n_val   = n_total - n_train
    if n_val < N_VAL_MIN or n_train < N_TRAIN_MIN:
        return None
    return n_train, n_val


# ---------------------------------------------------------------------------
# BOOTSTRAP BLOKOWY (stacjonarny, Politis-Romano 1994)
# Resampling BLOKAMI o sredniej dlugosci BLOCK_LEN zachowuje zaleznosc szeregowa
# i klastrowanie zmiennosci - czego HS (oraz bootstrap IID) NIE uwzglednia.
# Dzieki temu BOOT jest metoda jakosciowo rozna od symulacji historycznej.
# ---------------------------------------------------------------------------
def rolling_boot_var(log_returns: np.ndarray, dates: np.ndarray, alphas, seed_base=123):
    # Jeden resampling blokowy na dzien -> VaR dla wszystkich alf naraz.
    idx = dates[W:]

    def stat(x):
        qs = np.quantile(x, alphas, method="higher")
        return np.array([-q for q in qs])

    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        seed = make_seed(seed_base, t)
        boot = StationaryBootstrap(BLOCK_LEN, log_returns[t - W:t],
                                   seed=seed).apply(stat, reps=B)
        med = np.median(boot, axis=0)
        for i, a in enumerate(alphas):
            var_d[a].append(med[i])
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# MONTE CARLO — ZMIANA: stałe df=5 zamiast MLE-fitted df
# ---------------------------------------------------------------------------
def rolling_mc_var(log_returns: np.ndarray, dates: np.ndarray, alphas, seed_base=123):
    # Stale df=5; tylko loc i scale dopasowane do okna.
    # scale_t = sigma / sqrt(df/(df-2)) zapewnia Var(sim) = sigma^2.
    # Jedna symulacja na dzien -> VaR dla wszystkich alf; kwantyl "higher".
    idx = dates[W:]
    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t]
        mu = w.mean()
        sigma = w.std(ddof=1)
        scale_t = sigma / T_VAR_CORRECTION

        rng = np.random.default_rng(make_seed(seed_base, t))
        sim = mu + scale_t * rng.standard_t(T_DF, size=N)

        qs = np.quantile(sim, alphas, method="higher")
        for a, q in zip(alphas, qs):
            var_d[a].append(-q)
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# HISTORICAL — bez zmian
# (metoda nieparametryczna, nie zakłada rozkładu)
# ---------------------------------------------------------------------------
def rolling_hs_var(log_returns: np.ndarray, dates: np.ndarray, alphas):
    # Metoda nieparametryczna; VaR (empiryczny kwantyl) dla wszystkich alf.
    idx = dates[W:]
    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t]
        qs = np.quantile(w, alphas, method="higher")
        for a, q in zip(alphas, qs):
            var_d[a].append(-q)
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# VARIANCE-COVARIANCE (delta-normal) — klasyczny benchmark z rozkladem NORMALNYM
# ---------------------------------------------------------------------------
def rolling_vc_var(log_returns: np.ndarray, dates: np.ndarray, alphas):
    # Klasyczna specyfikacja RiskMetrics: kwantyl rozkladu NORMALNEGO (Z_NORM).
    # VaR = -(mu + sigma * Z_NORM[a]). Swiadomie "naiwny" benchmark bez grubych
    # ogonow -> kontrast z metodami t(5) (MC/GARCH/AG) i warunkowej wariancji.
    idx = dates[W:]
    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t]
        mu = w.mean()
        sig = w.std(ddof=1)
        for a in alphas:
            var_d[a].append(-(mu + sig * Z_NORM[a]))
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# GARCH(1,1) — ZMIANA: norm.ppf → t_dist.ppf(df=5) ze skalowaniem
# ---------------------------------------------------------------------------
def rolling_garch_var(log_returns: np.ndarray, dates: np.ndarray, alphas):
    # GARCH(1,1) estymuje mu i sigma_{T+1}; kwantyl reszt z t(df=5) (Z_T5),
    # spojne z Basel III ("GARCH + grube ogony"). Dopasowanie RAZ na dzien.
    idx = dates[W:]
    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t] * scale
        try:
            am = arch_model(w, mean="Constant", vol="GARCH", p=1, q=1,
                            dist="normal", rescale=False)
            res = am.fit(disp="off")
            fc = res.forecast(horizon=1, reindex=False)

            mu_f = fc.mean.iloc[-1, 0] / scale
            sigma_f = np.sqrt(fc.variance.iloc[-1, 0]) / scale

            for a in alphas:
                var_d[a].append(-(mu_f + sigma_f * Z_T5[a]))
        except Exception:
            for a in alphas:
                var_d[a].append(np.nan)
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# ARMA(1,1)+GARCH(1,1) — ZMIANA: norm.ppf → t_dist.ppf(df=5) ze skalowaniem
# ---------------------------------------------------------------------------
def rolling_arma_garch_var(log_returns: np.ndarray, dates: np.ndarray, alphas):
    # ARMA(1,1) -> mu_f, GARCH(1,1) reszt -> sigma_f; kwantyl reszt z t(df=5).
    # Dopasowanie RAZ na dzien dla wszystkich alf.
    idx = dates[W:]
    var_d = {a: [] for a in alphas}
    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t]
        try:
            arma_res = ARIMA(w, order=(1, 0, 1)).fit()
            # Odporne pobranie 1-krokowej prognozy (Series lub ndarray, zaleznie od wersji)
            mu_f = float(np.asarray(arma_res.forecast(steps=1)).ravel()[0])

            eps = arma_res.resid * scale
            garch_res = arch_model(eps, mean="Zero", vol="GARCH", p=1, q=1,
                                   dist="normal", rescale=False).fit(disp="off")
            garch_fc = garch_res.forecast(horizon=1, reindex=False)
            sigma_f = np.sqrt(garch_fc.variance.iloc[-1, 0]) / scale

            for a in alphas:
                var_d[a].append(-(mu_f + sigma_f * Z_T5[a]))
        except Exception:
            for a in alphas:
                var_d[a].append(np.nan)
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# GFQR (GARCH-Filtered Quantile Regression) — w kodzie dawniej "SVQR"
# (QuantileRegressor minimalizuje pinball loss bezpośrednio,
#  nie zakłada żadnego rozkładu parametrycznego — t-Student nie dotyczy)
# ---------------------------------------------------------------------------
def rolling_gfqr_var(log_returns: np.ndarray, dates: np.ndarray, alphas):
    # GARCH standaryzuje reszty RAZ na dzien (niezalezne od alpha),
    # regresja kwantylowa (QuantileRegressor) osobno dla kazdego alpha.
    idx = dates[W:]
    var_d = {a: [] for a in alphas}

    def _append_nan():
        for a in alphas:
            var_d[a].append(np.nan)

    for t in range(W, len(log_returns)):
        w = log_returns[t - W:t]

        try:
            res_g = arch_model(w * scale, mean="Constant", vol="GARCH",
                               p=1, q=1, dist="normal", rescale=False).fit(disp="off")
            fc_g = res_g.forecast(horizon=1, reindex=False)
            mu_garch = res_g.params["mu"] / scale
            sigma_next = np.sqrt(fc_g.variance.iloc[-1, 0]) / scale
            cond_vol = res_g.conditional_volatility / scale
            eps = (w - mu_garch) / np.where(cond_vol > 1e-8, cond_vol, 1e-8)
        except Exception:
            mu_garch = w.mean()
            sigma_next = w.std(ddof=1)
            eps = (w - mu_garch) / (sigma_next if sigma_next > 1e-8 else 1e-8)

        X_all, y_all = build_supervised(eps, lookback)

        split = split_sizes(len(X_all)) if len(X_all) >= N_TRAIN_MIN + N_VAL_MIN else None
        if split is None:
            _append_nan()
            continue

        n_train, _ = split
        X_tr, y_tr = X_all[:n_train], y_all[:n_train]
        X_val, y_val = X_all[n_train:], y_all[n_train:]
        x_pred = eps[-lookback:].reshape(1, -1).astype(np.float64)

        # Skalery walidacyjne (na train) i finalne (na pelnym oknie) - niezalezne od alpha
        x_scaler = StandardScaler().fit(X_tr)
        X_tr_s = x_scaler.transform(X_tr)
        X_val_s = x_scaler.transform(X_val)
        y_scaler = StandardScaler().fit(y_tr.reshape(-1, 1))
        y_tr_s = y_scaler.transform(y_tr.reshape(-1, 1)).ravel()

        x_scaler_f = StandardScaler().fit(X_all)
        y_scaler_f = StandardScaler().fit(y_all.reshape(-1, 1))
        X_all_s = x_scaler_f.transform(X_all)
        y_all_s = y_scaler_f.transform(y_all.reshape(-1, 1)).ravel()
        x_pred_s = x_scaler_f.transform(x_pred)

        for a in alphas:
            best_score = np.inf
            best_alpha_qr = grid_alpha_qr[0]
            for aq in grid_alpha_qr:
                try:
                    qr = QuantileRegressor(quantile=a, alpha=aq, solver="highs")
                    qr.fit(X_tr_s, y_tr_s)
                    y_val_hat_s = qr.predict(X_val_s)
                    y_val_hat = y_scaler.inverse_transform(y_val_hat_s.reshape(-1, 1)).ravel()
                    score = pinball_loss(y_val, y_val_hat, alpha=a)
                    if score < best_score:
                        best_score = score
                        best_alpha_qr = aq
                except Exception:
                    continue

            try:
                qr_final = QuantileRegressor(quantile=a, alpha=best_alpha_qr, solver="highs")
                qr_final.fit(X_all_s, y_all_s)
                q_eps_s = qr_final.predict(x_pred_s)[0]
                q_eps = y_scaler_f.inverse_transform([[q_eps_s]])[0, 0]
                var_d[a].append(-(mu_garch + q_eps * sigma_next))
            except Exception:
                var_d[a].append(np.nan)
    return {a: pd.Series(var_d[a], index=idx) for a in alphas}


# ---------------------------------------------------------------------------
# DANE + OBLICZENIA DLA WSZYSTKICH TICKEROW (zrownoleglenie po tickerach)
# ---------------------------------------------------------------------------
# 1) Pobierz dane wszystkich tickerow RAZ (workery nie korzystaja z sieci).
#    auto_adjust=True -> ceny skorygowane o splity i dywidendy.
raw = yf.download(tickers, start=DATE_START, end=DATE_END, interval="1d",
                  auto_adjust=True, progress=False, group_by="ticker")

data = {}   # ticker -> (log_r, dates)
for tk in tickers:
    try:
        sub = raw[tk] if isinstance(raw.columns, pd.MultiIndex) else raw
        close = sub["Close"].dropna()
    except Exception:
        close = pd.Series(dtype=float)
    if len(close) <= W:
        print(f"[POMIJAM] {tk}: za malo danych (len={len(close)}, W={W})")
        continue
    log_r = np.log(close).diff().dropna().to_numpy(dtype=np.float64)
    dts = close.index[1:].to_numpy()
    data[tk] = (log_r, dts)

tickers_ok = list(data.keys())
print(f"Tickery z danymi: {len(tickers_ok)}/{len(tickers)}")

# 2) Pojedyncze zadanie = jedna METODA dla jednej SPOLKI (liczy wszystkie alfy naraz).
METHODS_FNS = [
    ("BOOT",  rolling_boot_var),
    ("MC",    rolling_mc_var),
    ("HS",    rolling_hs_var),
    ("VC",    rolling_vc_var),
    ("GARCH", rolling_garch_var),
    ("AG",    rolling_arma_garch_var),
    ("GFQR",  rolling_gfqr_var),
]

def run_one(ticker, method_name, fn, log_r, dates):
    vd = fn(log_r, dates, alphas)
    cols = {f"{ticker}_{method_name}_a{a:.3f}": vd[a] for a in alphas}
    return ticker, method_name, cols

tasks = [(tk, name, fn) for tk in tickers_ok for (name, fn) in METHODS_FNS]

# 3) Log postepu do PLIKU (widoczny na zywo nawet pod nbconvert, gdzie tqdm jest
#    przechwytywany). Podglad na zywo w osobnym oknie:
#       Get-Content .\postep_obliczen.log -Wait -Tail 20
PROGRESS_LOG = "postep_obliczen.log"

def log_progress(msg, reset=False):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}"
    print(line, flush=True)
    with open(PROGRESS_LOG, "w" if reset else "a", encoding="utf-8") as _lf:
        _lf.write(line + "\n")

# 4) Zrownoleglenie po zadaniach (spolka x metoda). Wyniki splywaja w kolejnosci
#    ukonczenia (return_as="generator_unordered"). Postep: pasek tqdm (Run All)
#    ORAZ zapis do pliku logu (nbconvert / tlo).
log_progress(f"START komorki 0: {len(tasks)} zadan "
             f"({len(tickers_ok)} spolek x {len(METHODS_FNS)} metod x {len(alphas)} alf)",
             reset=True)

cols_all = {tk: {} for tk in tickers_ok}
done_by_ticker = {tk: 0 for tk in tickers_ok}
t0 = time.time()

gen = Parallel(n_jobs=N_JOBS, max_nbytes=None, return_as="generator_unordered")(
    delayed(run_one)(tk, name, fn, data[tk][0], data[tk][1])
    for (tk, name, fn) in tasks
)
pbar = tqdm(total=len(tasks), desc="Obliczanie VaR", ncols=100)
completed = 0
for tk, method_name, cols in gen:
    cols_all[tk].update(cols)
    done_by_ticker[tk] += 1
    completed += 1
    pbar.set_postfix_str(f"{tk}/{method_name}")
    pbar.update(1)

    elapsed = time.time() - t0
    eta = elapsed / completed * (len(tasks) - completed)
    tickers_done = sum(1 for v in done_by_ticker.values() if v == len(METHODS_FNS))
    log_progress(f"[{completed}/{len(tasks)}] zadanie OK: spolka={tk} metoda={method_name} "
                 f"| spolki ukonczone={tickers_done}/{len(tickers_ok)} "
                 f"| czas={elapsed/60:.1f} min | ETA~{eta/60:.1f} min")
pbar.close()
log_progress("KONIEC obliczen VaR (komorka 0) - trwa skladanie i zapis wynikow.")

# 5) Zlozenie wynikow per spolka (kolumny: {spolka}_{metoda}_a{alfa}).
out_by_ticker = {tk: pd.concat(cols_all[tk], axis=1).sort_index() for tk in tickers_ok}
out = pd.concat([out_by_ticker[tk] for tk in tickers_ok], axis=1).sort_index()
realized_by_ticker = {
    tk: pd.Series(data[tk][0], index=pd.DatetimeIndex(data[tk][1]))
    for tk in tickers_ok
}
methods = ["BOOT", "MC", "HS", "VC", "GARCH", "AG", "GFQR"]
tickers = tickers_ok   # dalsze komorki uzywaja tylko tickerow z danymi

# 6) Podsumowanie + diagnostyka NaN (NaN = niepowodzenie dopasowania lub za malo danych)
print(f"\nGotowe. out.shape = {out.shape} "
      f"(tickery x metody x alfy = {len(tickers_ok)} x 7 x {len(alphas)})")
for tk in tickers_ok:
    o = out_by_ticker[tk]
    print(f"  {tk}: shape={o.shape}, NaN={int(o.isna().sum().sum())}")

# 7) Zapis wynikow VaR na dysk -> szybkie iteracje komorek 1/2 bez ponownego liczenia.
with open("var_results.pkl", "wb") as _f:
    pickle.dump({"out_by_ticker": out_by_ticker, "realized_by_ticker": realized_by_ticker,
                 "tickers": tickers, "alphas": alphas, "methods": methods,
                 "QUICK": QUICK}, _f)
print("Zapisano wyniki VaR do var_results.pkl")
log_progress("Zapisano var_results.pkl - KOMORKA 0 ZAKONCZONA.")

In [ ]:
# wykresy z oknami i excelc export
import time
import numpy as np
import pandas as pd
from scipy.stats import chi2

def _log(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}"
    print(line, flush=True)
    with open("postep_obliczen.log", "a", encoding="utf-8") as _lf:
        _lf.write(line + "\n")

def is_reject(v):
    # Odporne na numpy.bool_ i NaN (porownanie 'is True' zawodzi dla numpy.bool_).
    return isinstance(v, (bool, np.bool_)) and bool(v)
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------------------
# Wejscie: obiekty z komorki 0. Jesli brak w pamieci (np. swiezy kernel) -
# wczytaj z var_results.pkl zapisanego przez komorke 0.
# ---------------------------------------------------------------------------
if "out_by_ticker" not in globals():
    import pickle
    with open("var_results.pkl", "rb") as _f:
        _d = pickle.load(_f)
    out_by_ticker = _d["out_by_ticker"]
    realized_by_ticker = _d["realized_by_ticker"]
    tickers = _d["tickers"]
    alphas = _d["alphas"]
    out = pd.concat([out_by_ticker[tk] for tk in tickers], axis=1).sort_index()
    print("Wczytano wyniki VaR z var_results.pkl")

for _n in ("out", "realized_by_ticker", "tickers", "alphas"):
    if _n not in globals():
        raise NameError(f"Brak '{_n}'. Uruchom komorke 0 albo zapewnij var_results.pkl.")
alphas = sorted(alphas)
methods = ["BOOT", "MC", "HS", "VC", "GARCH", "AG", "GFQR"]
N_PARTS = 4   # podzial okresu testowego na 4 ROWNE podokresy

print(f"Tickery : {tickers}")
print(f"Alfa    : {alphas}")
print(f"Metody  : {methods}\n")

# ---------------------------------------------------------------------------
# Funkcje testowe
# ---------------------------------------------------------------------------
def prepare_hits(realized, var_series):
    tmp = pd.concat([realized.rename("r"), var_series.rename("VaR")], axis=1).dropna()
    hits = (tmp["r"] < -tmp["VaR"]).astype(int)
    return tmp, hits

def kupiec_pof_test(realized, var_series, alpha):
    tmp, hits = prepare_hits(realized, var_series)
    T = len(hits)
    x = int(hits.sum())
    if T < 2:
        return x, np.nan, np.nan
    p_hat = np.clip(x / T, 1e-12, 1 - 1e-12)
    a = np.clip(alpha, 1e-12, 1 - 1e-12)
    LR = -2.0 * (
        (T - x) * np.log(1 - a) + x * np.log(a) -
        (T - x) * np.log(1 - p_hat) - x * np.log(p_hat)
    )
    return x, float(LR), float(chi2.sf(LR, df=1))

def christoffersen_ind_test(realized, var_series, alpha):
    tmp, hits = prepare_hits(realized, var_series)
    T = len(hits)
    x = int(hits.sum())
    if T < 2:
        return x, np.nan, np.nan
    h = hits.to_numpy(dtype=int)
    h_prev, h_curr = h[:-1], h[1:]
    N00 = int(((h_prev == 0) & (h_curr == 0)).sum())
    N01 = int(((h_prev == 0) & (h_curr == 1)).sum())
    N10 = int(((h_prev == 1) & (h_curr == 0)).sum())
    N11 = int(((h_prev == 1) & (h_curr == 1)).sum())
    def cp(p): return np.clip(p, 1e-12, 1 - 1e-12)
    p_hat = cp(x / T)
    p01 = cp(N01 / (N00 + N01)) if (N00 + N01) > 0 else cp(0.0)
    p11 = cp(N11 / (N10 + N11)) if (N10 + N11) > 0 else cp(0.0)
    LR_ind = -2.0 * (
        (N00 + N10) * np.log(1 - p_hat) + (N01 + N11) * np.log(p_hat) -
        N00 * np.log(1 - p01) - N01 * np.log(p01) -
        N10 * np.log(1 - p11) - N11 * np.log(p11)
    )
    return x, float(LR_ind), float(chi2.sf(LR_ind, df=1))

def pinball_loss_score(realized, var_series, alpha):
    tmp = pd.concat([realized.rename("r"), var_series.rename("VaR")], axis=1).dropna()
    q_hat = -tmp["VaR"]
    e = tmp["r"] - q_hat
    return float(np.maximum(alpha * e, (alpha - 1.0) * e).mean())

def perignon_smith_test(realized, var_by_alpha):
    available = sorted(var_by_alpha.keys())
    if len(available) < 2:
        return np.nan, np.nan
    alpha_desc = sorted(available, reverse=True)
    tmp = pd.concat(
        [realized.rename("r")] +
        [var_by_alpha[a].rename(f"VaR_{a:.3f}") for a in alpha_desc],
        axis=1
    ).dropna()
    if tmp.empty:
        return np.nan, np.nan
    L = (-tmp["r"]).to_numpy(dtype=np.float64)
    V = np.column_stack([tmp[f"VaR_{a:.3f}"].to_numpy(dtype=np.float64) for a in alpha_desc])
    V = np.sort(V, axis=1)
    K = len(alpha_desc)
    probs = [1.0 - alpha_desc[0]]
    for i in range(K - 1):
        probs.append(alpha_desc[i] - alpha_desc[i + 1])
    probs.append(alpha_desc[-1])
    probs = np.asarray(probs, dtype=np.float64)
    counts = np.zeros(K + 1, dtype=int)
    counts[0] = int((L <= V[:, 0]).sum())
    for i in range(1, K):
        counts[i] = int(((L > V[:, i - 1]) & (L <= V[:, i])).sum())
    counts[K] = int((L > V[:, K - 1]).sum())
    T = counts.sum()
    if T == 0:
        return np.nan, np.nan
    p_hat = counts / T
    mask = counts > 0
    LR = 2.0 * np.sum(counts[mask] * np.log(p_hat[mask] / probs[mask]))
    return float(LR), float(chi2.sf(LR, df=K))

# ---------------------------------------------------------------------------
# Funkcja budująca tabele wynikowe dla danego slice'a danych
# ---------------------------------------------------------------------------
def run_verification(realized_slice, out_slice):
    # Pérignon-Smith
    ps_rows = []
    for method in methods:
        var_by_alpha = {}
        for a in alphas:
            col = f"{ticker}_{method}_a{a:.3f}"
            if col in out_slice.columns:
                var_by_alpha[a] = out_slice[col]
        ps_lr, ps_p = perignon_smith_test(realized_slice, var_by_alpha)
        ps_rows.append({
            "Method": method,
            "PS_LR": round(ps_lr, 4) if not np.isnan(ps_lr) else np.nan,
            "PS_p": round(ps_p, 4) if not np.isnan(ps_p) else np.nan,
            "PS_reject": (ps_p < 0.05) if not np.isnan(ps_p) else np.nan,
        })
    ps_df = pd.DataFrame(ps_rows).set_index("Method")

    # Per-alpha tabele
    alpha_dfs = {}
    for a in alphas:
        rows = []
        for method in methods:
            col = f"{ticker}_{method}_a{a:.3f}"
            if col not in out_slice.columns:
                continue
            var_series = out_slice[col]
            viol_k, _, kupiec_p = kupiec_pof_test(realized_slice, var_series, a)
            viol_c, _, ind_p = christoffersen_ind_test(realized_slice, var_series, a)
            loss = pinball_loss_score(realized_slice, var_series, a)
            expected = int(round(a * len(var_series.dropna())))
            rows.append({
                "Method": method,
                "Violations": viol_k,
                "Expected_viol": expected,
                "Kupiec_p": round(kupiec_p, 4) if not np.isnan(kupiec_p) else np.nan,
                "Kupiec_reject": (kupiec_p < 0.05) if not np.isnan(kupiec_p) else np.nan,
                "Christ_IND_p": round(ind_p, 4) if not np.isnan(ind_p) else np.nan,
                "Christ_reject": (ind_p < 0.05) if not np.isnan(ind_p) else np.nan,
                "Pinball_loss": round(loss, 6) if not np.isnan(loss) else np.nan,
            })
        alpha_dfs[a] = pd.DataFrame(rows).set_index("Method")

    return ps_df, alpha_dfs

# ---------------------------------------------------------------------------
# (Obliczenia weryfikacji + zapis do Excela sa nizej, w petli po tickerach.)
# Szczegolowe tabele trafiaja do Excela; konsola pokazuje tylko postep.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Export do Excel
# ---------------------------------------------------------------------------
COLS_DETAIL = ["Violations", "Expected_viol", "Kupiec_p", "Kupiec_reject",
               "Christ_IND_p", "Christ_reject", "Pinball_loss"]
COLS_PS     = ["PS_LR", "PS_p", "PS_reject"]

# Kolory
CLR_HEADER_DARK  = "1F3864"   # granatowy — nagłówki główne
CLR_HEADER_MID   = "2E75B6"   # niebieski — nagłówki sekcji
CLR_HEADER_LIGHT = "D6E4F0"   # jasnoniebieski — nagłówki kolumn
CLR_REJECT_TRUE  = "FCE4D6"   # łososiowy — odrzucenie H0
CLR_SECTION_BG   = "F2F2F2"   # szary — nagłówek okresu
CLR_WHITE        = "FFFFFF"

FONT_MID   = Font(name="Arial", bold=True, color="FFFFFF", size=10)
FONT_LIGHT = Font(name="Arial", bold=True, color="1F3864", size=9)
FONT_BODY  = Font(name="Arial", size=9)
FONT_SEC   = Font(name="Arial", bold=True, color="1F3864", size=9)

ALIGN_CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALIGN_LEFT   = Alignment(horizontal="left",   vertical="center")

thin = Side(style="thin", color="BFBFBF")
BORDER_THIN = Border(left=thin, right=thin, top=thin, bottom=thin)

def fill(hex_color):
    return PatternFill("solid", start_color=hex_color, fgColor=hex_color)

def write_ps_table(ws, start_row, ps_df, label):
    """Zapisuje tabelę Pérignon-Smith, zwraca nowy start_row."""
    # Nagłówek sekcji
    ws.cell(row=start_row, column=1, value=label).font = FONT_SEC
    ws.cell(row=start_row, column=1).fill = fill(CLR_SECTION_BG)
    ws.merge_cells(start_row=start_row, start_column=1,
                   end_row=start_row, end_column=1 + len(COLS_PS))
    ws.cell(row=start_row, column=1).alignment = ALIGN_LEFT
    start_row += 1

    # Podtytuł PS
    ws.cell(row=start_row, column=1, value="Pérignon & Smith (2008)").font = FONT_MID
    ws.cell(row=start_row, column=1).fill = fill(CLR_HEADER_MID)
    ws.merge_cells(start_row=start_row, start_column=1,
                   end_row=start_row, end_column=1 + len(COLS_PS))
    ws.cell(row=start_row, column=1).alignment = ALIGN_LEFT
    ws.cell(row=start_row, column=1).font = FONT_MID
    start_row += 1

    # Nagłówki kolumn
    headers = ["Method"] + COLS_PS
    for ci, h in enumerate(headers, 1):
        c = ws.cell(row=start_row, column=ci, value=h)
        c.font = FONT_LIGHT
        c.fill = fill(CLR_HEADER_LIGHT)
        c.alignment = ALIGN_CENTER
        c.border = BORDER_THIN
    start_row += 1

    # Dane
    for method, row_data in ps_df.iterrows():
        reject_val = row_data.get("PS_reject", False)
        row_fill = fill(CLR_REJECT_TRUE) if is_reject(reject_val) else fill(CLR_WHITE)
        ws.cell(row=start_row, column=1, value=method).font = FONT_BODY
        ws.cell(row=start_row, column=1).fill = row_fill
        ws.cell(row=start_row, column=1).border = BORDER_THIN
        ws.cell(row=start_row, column=1).alignment = ALIGN_LEFT
        for ci, col in enumerate(COLS_PS, 2):
            val = row_data.get(col, "")
            if isinstance(val, (bool, np.bool_)):
                val = "TAK" if bool(val) else "NIE"
            elif isinstance(val, float) and np.isnan(val):
                val = ""
            c = ws.cell(row=start_row, column=ci, value=val)
            c.font = FONT_BODY
            c.fill = row_fill
            c.alignment = ALIGN_CENTER
            c.border = BORDER_THIN
        start_row += 1

    return start_row + 1  # odstęp

def write_alpha_table(ws, start_row, alpha_df, alpha_val):
    """Zapisuje tabelę per-alpha, zwraca nowy start_row."""
    # Nagłówek alpha
    hdr = f"alpha = {alpha_val}"
    ws.cell(row=start_row, column=1, value=hdr).font = FONT_MID
    ws.cell(row=start_row, column=1).fill = fill(CLR_HEADER_MID)
    ws.merge_cells(start_row=start_row, start_column=1,
                   end_row=start_row, end_column=1 + len(COLS_DETAIL))
    ws.cell(row=start_row, column=1).alignment = ALIGN_LEFT
    ws.cell(row=start_row, column=1).font = FONT_MID
    start_row += 1

    # Nagłówki kolumn
    headers = ["Method"] + COLS_DETAIL
    for ci, h in enumerate(headers, 1):
        c = ws.cell(row=start_row, column=ci, value=h)
        c.font = FONT_LIGHT
        c.fill = fill(CLR_HEADER_LIGHT)
        c.alignment = ALIGN_CENTER
        c.border = BORDER_THIN
    start_row += 1

    # Dane
    for method, row_data in alpha_df.iterrows():
        reject_val = row_data.get("Christ_reject", False)
        kupiec_rej = row_data.get("Kupiec_reject", False)
        any_reject = is_reject(reject_val) or is_reject(kupiec_rej)
        row_fill = fill(CLR_REJECT_TRUE) if any_reject else fill(CLR_WHITE)

        ws.cell(row=start_row, column=1, value=method).font = FONT_BODY
        ws.cell(row=start_row, column=1).fill = row_fill
        ws.cell(row=start_row, column=1).border = BORDER_THIN
        ws.cell(row=start_row, column=1).alignment = ALIGN_LEFT

        for ci, col in enumerate(COLS_DETAIL, 2):
            val = row_data.get(col, "")
            if isinstance(val, (bool, np.bool_)):
                val = "TAK" if bool(val) else "NIE"
            elif isinstance(val, float) and np.isnan(val):
                val = ""
            c = ws.cell(row=start_row, column=ci, value=val)
            c.font = FONT_BODY
            c.fill = row_fill
            c.alignment = ALIGN_CENTER
            c.border = BORDER_THIN
        start_row += 1

    return start_row + 1

def build_sheet(wb, sheet_name, alpha_val, full_ps, full_alpha_dfs, window_results):
    ws = wb.create_sheet(title=sheet_name)

    # Szerokości kolumn
    ws.column_dimensions["A"].width = 10
    for ci in range(2, 12):
        ws.column_dimensions[get_column_letter(ci)].width = 16

    # Tytuł arkusza
    title_val = f"Weryfikacja VaR — {ticker} | alpha = {alpha_val}"
    ws.cell(row=1, column=1, value=title_val)
    ws.cell(row=1, column=1).font = Font(name="Arial", bold=True, color="FFFFFF", size=12)
    ws.cell(row=1, column=1).fill = fill(CLR_HEADER_DARK)
    ws.cell(row=1, column=1).alignment = ALIGN_LEFT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=10)
    ws.row_dimensions[1].height = 22

    current_row = 3

    # --- Cały okres ---
    current_row = write_ps_table(ws, current_row, full_ps, full_label)
    current_row = write_alpha_table(ws, current_row, full_alpha_dfs[alpha_val], alpha_val)
    current_row += 1  # dodatkowy odstęp

    # --- Okna ---
    for lbl, ps_df, alpha_dfs in window_results:
        current_row = write_ps_table(ws, current_row, ps_df, lbl)
        if alpha_val in alpha_dfs:
            current_row = write_alpha_table(ws, current_row, alpha_dfs[alpha_val], alpha_val)
        current_row += 1

    # Freeze panes
    ws.freeze_panes = "A3"

    return ws

# ---------------------------------------------------------------------------
# Budowa Excela: arkusze per ticker x alfa + arkusz PODSUMOWANIE.
# Okres testowy dzielony na N_PARTS ROWNYCH podokresow (+ caly okres).
# ---------------------------------------------------------------------------
wb = Workbook()
wb.remove(wb.active)

# Akumulator odrzucen do tabeli zbiorczej (liczone na calym okresie, po tickerach)
REJECT_TESTS = {"Kupiec": "Kupiec_reject", "ChristIND": "Christ_reject"}
reject_counts = {t: {a: {m: 0 for m in methods} for a in alphas} for t in REJECT_TESTS}
ps_reject_counts = {m: 0 for m in methods}   # PS jest niezalezne od alfa
n_tickers_done = 0

def equal_windows(n_total, n_parts, index):
    """Dzieli [0, n_total) na n_parts mozliwie rownych czesci."""
    bounds = [round(k * n_total / n_parts) for k in range(n_parts + 1)]
    ws_list = []
    for k in range(n_parts):
        s, e = bounds[k], bounds[k + 1]
        if e <= s:
            continue
        s0 = str(index[s])[:10]
        s1 = str(index[e - 1])[:10]
        ws_list.append((f"Podokres {k + 1}/{n_parts}: {s0} - {s1} ({e - s} obs.)", s, e))
    return ws_list

_log(f"START komorki 1 (backtesting + Excel): {len(tickers)} spolek.")
for ticker in tickers:
    if ticker not in realized_by_ticker:
        continue
    realized_log = realized_by_ticker[ticker]
    common_idx = out.index.intersection(realized_log.dropna().index)
    out_aligned = out.loc[common_idx]
    real_aligned = realized_log.loc[common_idx]
    n_total = len(common_idx)

    windows = equal_windows(n_total, N_PARTS, common_idx)

    full_label = (f"Caly okres {ticker}: {str(common_idx[0])[:10]} - "
                  f"{str(common_idx[-1])[:10]} ({n_total} obs.)")
    full_ps, full_alpha_dfs = run_verification(real_aligned, out_aligned)
    window_results = []
    for lbl, s, e in windows:
        ps_df, alpha_dfs = run_verification(real_aligned.iloc[s:e], out_aligned.iloc[s:e])
        window_results.append((lbl, ps_df, alpha_dfs))

    # akumulacja odrzucen (caly okres) do tabeli zbiorczej
    for m in methods:
        if m in full_ps.index and is_reject(full_ps.loc[m, "PS_reject"]):
            ps_reject_counts[m] += 1
    for t, col in REJECT_TESTS.items():
        for a in alphas:
            adf = full_alpha_dfs.get(a)
            if adf is None:
                continue
            for m in methods:
                if m in adf.index and is_reject(adf.loc[m, col]):
                    reject_counts[t][a][m] += 1
    n_tickers_done += 1

    for a_val in alphas:
        build_sheet(wb, f"{ticker}_a{a_val}", a_val, full_ps, full_alpha_dfs, window_results)
    print(f"[{ticker}] gotowe: {n_total} obs., {len(windows)} podokresow")
    _log(f"EXCEL [{n_tickers_done}/{len(tickers)}] arkusze gotowe: spolka={ticker}")

# ---------------------------------------------------------------------------
# Tabela zbiorcza: ile razy metoda zostala ODRZUCONA (caly okres) na N tickerach
# ---------------------------------------------------------------------------
summary_cols = {}
for t in REJECT_TESTS:
    for a in alphas:
        summary_cols[f"{t}_a{a:.3f}"] = {m: reject_counts[t][a][m] for m in methods}
summary_cols["PS"] = {m: ps_reject_counts[m] for m in methods}
summary_df = pd.DataFrame(summary_cols).reindex(methods)
summary_df["SUMA"] = summary_df.sum(axis=1)

print(f"\n===== PODSUMOWANIE: liczba ODRZUCEN na {n_tickers_done} tickerach (caly okres) =====")
print("(mniej = lepiej; kolumny = test x alfa, PS = Perignon-Smith)")
print(summary_df.to_string())
if n_tickers_done > 0:
    print("\nNajlepszy (najmniej odrzucen, wg SUMA):", summary_df["SUMA"].idxmin())
    print("Najgorszy (najwiecej odrzucen, wg SUMA):", summary_df["SUMA"].idxmax())

# arkusz PODSUMOWANIE (na poczatku skoroszytu), min=zielony / max=lososiowy w kolumnie
ws_sum = wb.create_sheet(title="PODSUMOWANIE", index=0)
ws_sum.cell(row=1, column=1,
            value=f"Liczba odrzucen H0 na {n_tickers_done} tickerach (caly okres) - mniej = lepiej")
ws_sum.cell(row=1, column=1).font = Font(name="Arial", bold=True, color="FFFFFF", size=12)
ws_sum.cell(row=1, column=1).fill = fill(CLR_HEADER_DARK)
ws_sum.merge_cells(start_row=1, start_column=1, end_row=1, end_column=1 + len(summary_df.columns))
ws_sum.row_dimensions[1].height = 22
ws_sum.column_dimensions["A"].width = 10
for ci, h in enumerate(["Method"] + list(summary_df.columns), 1):
    c = ws_sum.cell(row=3, column=ci, value=h)
    c.font = FONT_LIGHT; c.fill = fill(CLR_HEADER_LIGHT)
    c.alignment = ALIGN_CENTER; c.border = BORDER_THIN
    ws_sum.column_dimensions[get_column_letter(ci)].width = 14
r = 4
for m in methods:
    ws_sum.cell(row=r, column=1, value=m).font = FONT_BODY
    ws_sum.cell(row=r, column=1).border = BORDER_THIN
    for ci, col in enumerate(summary_df.columns, 2):
        v = int(summary_df.loc[m, col])
        cc = ws_sum.cell(row=r, column=ci, value=v)
        cc.font = FONT_BODY; cc.alignment = ALIGN_CENTER; cc.border = BORDER_THIN
        colmin = int(summary_df[col].min()); colmax = int(summary_df[col].max())
        if colmin != colmax and v == colmin:
            cc.fill = fill("E2EFDA")
        elif colmin != colmax and v == colmax:
            cc.fill = fill(CLR_REJECT_TRUE)
    r += 1
ws_sum.freeze_panes = "B4"

output_path = "var_verification_all.xlsx"
try:
    wb.save(output_path)
    print(f"\nZapisano: {output_path} ({len(wb.sheetnames)} arkuszy, w tym PODSUMOWANIE)")
    _log(f"KOMORKA 1 ZAKONCZONA - zapisano {output_path} ({len(wb.sheetnames)} arkuszy).")
except PermissionError:
    alt_path = "var_verification_all_NOWY.xlsx"
    wb.save(alt_path)
    print(f"\n[UWAGA] {output_path} byl otwarty/zablokowany - zapisano jako {alt_path}.")
    _log(f"KOMORKA 1 ZAKONCZONA - {output_path} zablokowany, zapisano jako {alt_path}.")

In [ ]:
# wykresy: VaR vs strata
# - dla KAZDEGO tickera: 1 wykres (caly okres) dla jednej alfy = PLOT_ALPHA
# - dla FULL_PLOT_TICKER (np. AAPL): dodatkowo caly okres + kazdy podokres x kazda alfa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PLOT_ALPHA = 0.01            # alfa dla zbiorczych wykresow "caly okres" kazdego tickera
FULL_PLOT_TICKER = "AAPL"    # ten ticker dostaje pelna siatke (okresy x alfy)
N_PARTS = 4

# wczytaj z pickle, jesli brak w pamieci (np. swiezy kernel)
if "out_by_ticker" not in globals():
    import pickle
    with open("var_results.pkl", "rb") as _f:
        _d = pickle.load(_f)
    out_by_ticker = _d["out_by_ticker"]; realized_by_ticker = _d["realized_by_ticker"]
    tickers = _d["tickers"]; alphas = _d["alphas"]; methods = _d["methods"]

METHOD_COLORS = {
    "BOOT": "#E63946", "MC": "#2196F3", "HS": "#FF9800", "VC": "#9C27B0",
    "GARCH": "#4CAF50", "AG": "#00BCD4", "GFQR": "#FF5722",
}
LOSS_COLOR = "#1a1a1a"

def _equal_windows(n_total, n_parts, index):
    bounds = [round(k * n_total / n_parts) for k in range(n_parts + 1)]
    out_w = []
    for k in range(n_parts):
        s, e = bounds[k], bounds[k + 1]
        if e > s:
            out_w.append((f"podokres {k + 1}/{n_parts}", s, e))
    return out_w

def _aligned(ticker):
    o = out_by_ticker[ticker]
    r = realized_by_ticker[ticker]
    ci = o.index.intersection(r.dropna().index)
    return o.loc[ci], r.loc[ci], ci

def plot_var_vs_loss(ticker, real_slice, out_slice, alpha, title):
    fig, ax = plt.subplots(figsize=(14, 5))
    loss = -real_slice
    ax.plot(loss.index, loss.values, color=LOSS_COLOR, linewidth=0.8,
            alpha=0.7, label="loss", zorder=2)
    for method in methods:
        col = f"{ticker}_{method}_a{alpha:.3f}"
        if col not in out_slice.columns:
            continue
        vs = out_slice[col].dropna()
        color = METHOD_COLORS.get(method, "#888888")
        ax.plot(vs.index, vs.values, color=color, linewidth=1.2, label=method, zorder=3)
        ax.plot(vs.index, -vs.values, color=color, linewidth=1.2,
                linestyle="--", alpha=0.4, zorder=3)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle=":")
    ax.set_title(f"{title}\nalpha = {alpha}", fontsize=11, fontweight="bold", pad=8)
    ax.set_ylabel("Log-stopa zwrotu / VaR", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12),
              ncol=min(len(methods) + 1, 5), fontsize=8, frameon=True, framealpha=0.9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

# 1) Dla KAZDEGO tickera (oprocz FULL_PLOT_TICKER): caly okres, jedna alfa
for ticker in tickers:
    if ticker not in out_by_ticker or ticker == FULL_PLOT_TICKER:
        continue
    o, r, ci = _aligned(ticker)
    plot_var_vs_loss(ticker, r, o, PLOT_ALPHA, f"{ticker} - caly okres ({len(ci)} obs.)")

# 2) FULL_PLOT_TICKER: caly okres + kazdy podokres, dla KAZDEJ alfy
if FULL_PLOT_TICKER in out_by_ticker:
    o, r, ci = _aligned(FULL_PLOT_TICKER)
    slices = [("caly okres", 0, len(ci))] + _equal_windows(len(ci), N_PARTS, ci)
    for lbl, s, e in slices:
        for a in alphas:
            plot_var_vs_loss(FULL_PLOT_TICKER, r.iloc[s:e], o.iloc[s:e], a,
                             f"{FULL_PLOT_TICKER} - {lbl} ({e - s} obs.)")
else:
    print(f"FULL_PLOT_TICKER={FULL_PLOT_TICKER} niedostepny w danych.")